# Caraway & Jain: per-country data collection

Period: **1 Jan 2024 – 31 May 2026**. Run one country per cell, in any order and on any day. Each finished country is saved in `outputs/countries/<Country>/`. Run **Combine** whenever you want the merged, month-sorted workbook.

The classification sheet (`.xlsx`, or the `General` and `Codewords` CSVs) must be in this folder next to `caraway_scraper.py`.

## 1. Setup (once per machine or Codespace), then restart the kernel

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Check the sheet is found and see each country's sources

In [5]:
from caraway_scraper import main
main(["--list"])

Sheet: /workspaces/Discourse_Analysis_Summer_Research/Sharvi/W/o blocking  |  98 keywords

USA
   [on ] wayback          Govt press release       https://www.state.gov/press-releases/
   [on ] wayback          Ministry of Environment  https://www.epa.gov/newsreleases/search
   [on ] wayback          Govt press release       https://2021-2025.state.gov/
   [on ] federal_register Executive order / Presid https://www.federalregister.gov/api/v1/documents.json

International (news)
   [on ] auto             News                     https://www.nytimes.com/section/climate
   [on ] auto             News                     https://www.nytimes.com/section/business/energy-environment
   [on ] auto             News                     https://www.washingtonpost.com/climate-environment/
   [on ] auto             News                     https://www.reuters.com/business/energy/
   [on ] auto             News                     https://www.reuters.com/business/environment/
   [on ] auto           

## 3. Optional: health-check one country before scraping it
Writes `outputs/countries/<Country>/probe_report.csv`. A status of `BLOCKED` means: run that country on your own computer (README).

In [6]:
main(["--country", "USA", "--probe"])

20:21:02 INFO USA: 4 sources, 98 keywords, 2024-01-01 to 2026-05-31
20:23:13 WARNING   Internet Archive CDX failed for www.state.gov/releases/ (failed)
20:23:13 INFO   https://www.state.gov/press-releases/                          no results (failed) (0 links)
20:25:34 WARNING   Internet Archive CDX failed for www.epa.gov/newsreleases/ (http_503)
20:25:34 INFO   https://www.epa.gov/newsreleases/search                        no results (http_503) (0 links)
20:26:56 WARNING   Internet Archive CDX failed for 2021-2025.state.gov/ (http_503)
20:26:56 INFO   https://2021-2025.state.gov/                                   no results (http_503) (0 links)
20:26:58 INFO   https://www.federalregister.gov/api/v1/documents.json          no results (http_400) (0 links)
20:26:58 INFO Probe report: outputs/countries/USA/probe_report.csv


## 4. Scrape each country
Each cell can take 30 minutes to a few hours. If it stops, run the same cell again: pages already fetched are cached, so it resumes. For a quick test first, add `"--limit-per-source", "20"`.

### USA
Run on your own computer. All three US websites block cloud servers; in Codespaces the Internet Archive and Federal Register sources still work.

In [ ]:
main(["--country", "USA"])

### Canada

In [ ]:
main(["--country", "Canada"])

### UK

In [ ]:
main(["--country", "UK"])

### EU
For the Commission press corner add "--js" (own computer).

In [ ]:
main(["--country", "EU"])

### New Zealand

In [ ]:
main(["--country", "New Zealand"])

### South Korea

In [ ]:
main(["--country", "South Korea"])

### China

In [ ]:
main(["--country", "China"])

### India
PIB is form-based: put its links in manual_urls/India.csv first (README, section 4).

In [ ]:
main(["--country", "India"])

### Singapore
sgpc.gov.sg needs "--js" (own computer).

In [ ]:
main(["--country", "Singapore"])

### Hong Kong
Slowest country: reads one index page per day (~880 days).

In [ ]:
main(["--country", "Hong Kong"])

### Optional: news outlets (kept separate from the official corpus)

In [ ]:
main(["--country", "International (news)"])

## 5. Combine every finished country into one workbook
Creates `outputs/Caraway_Jain_Articles_ALL.xlsx` (all articles, one tab per month, monthly counts by country with empty months shaded) plus `articles_all.csv` and `corpus_all.jsonl`.

In [ ]:
main(["--combine"])

## 6. Save results to GitHub (so they survive the Codespace)
Pushes the country folders and the combined files, not the cache. Click the printed links to download.

In [ ]:
import subprocess, datetime
from pathlib import Path

def push_outputs(paths=("outputs/countries", "outputs/Caraway_Jain_Articles_ALL.xlsx", "outputs/articles_all.csv")):
    paths = [p for p in paths if Path(p).exists()]
    subprocess.run(["git", "add", "-f", *paths], check=True)
    r = subprocess.run(["git", "commit", "-m", f"Scraper outputs {datetime.datetime.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[0] if r.stdout else r.stderr.strip())
    subprocess.run(["git", "pull", "--rebase"], check=False)   # pick up countries pushed from your laptop
    subprocess.run(["git", "push"], check=True)
    repo = subprocess.run(["git", "remote", "get-url", "origin"], capture_output=True, text=True).stdout.strip()
    repo = repo.removesuffix(".git").replace("git@github.com:", "https://github.com/")
    branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
    print(f"Files: {repo}/tree/{branch}/outputs")

push_outputs()